In [ ]:
!pip install -q transformers[sentencepiece] datasets accelerate evaluate sacrebleu bert_score nltk

In [ ]:
import pandas as pd
import numpy as np
import torch
import nltk
import re
import gc
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
import evaluate

nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)

import os
import shutil

working_dir = "/kaggle/working"
if os.path.exists(working_dir):
    for filename in os.listdir(working_dir):
        file_path = os.path.join(working_dir, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print(f"Failed to delete {file_path}: {e}")

print("Kaggle working directory cleaned!")

class Config:
    DATASET_PATH = "/kaggle/input/aacl-dialect-dataset/AACL dialect  - Sentence.csv"
    SOURCE_LANG, TARGET_LANG = "Sylhet Language", "Chittagong Language"
    MODEL_CHECKPOINT = "facebook/mbart-large-50-many-to-many-mmt"
    SOURCE_LANG_CODE, TARGET_LANG_CODE = "bn_IN", "bn_IN"
    OUTPUT_DIR = "./mbart-syl-to-ctg"
    MAX_INPUT_LENGTH, MAX_TARGET_LENGTH = 128, 128
    BATCH_SIZE, GRADIENT_ACCUMULATION_STEPS = 8, 2
    EVAL_STRATEGY = "epoch"
    NUM_EPOCHS, LEARNING_RATE = 25, 5e-5
    WEIGHT_DECAY, WARMUP_STEPS = 0.01, 50
    TRAIN_RATIO, VALIDATION_RATIO, TEST_RATIO = 0.7, 0.1, 0.2
    SEED = 42

config = Config()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

df = pd.read_csv(config.DATASET_PATH)
df = df[[config.SOURCE_LANG, config.TARGET_LANG]].rename(columns={config.SOURCE_LANG: 'syl', config.TARGET_LANG: 'ctg'})
df.dropna(inplace=True)

def clean_text(text):
    return text.strip() if isinstance(text, str) else ""
df['syl'] = df['syl'].apply(clean_text)
df['ctg'] = df['ctg'].apply(clean_text)
df = df[(df['syl'] != '') & (df['ctg'] != '')]
print(f"Dataset loaded with {len(df)} samples after cleaning.")

train_df, temp_df = train_test_split(df, test_size=(1 - config.TRAIN_RATIO), random_state=config.SEED)
val_df, test_df = train_test_split(temp_df, test_size=(config.TEST_RATIO / (config.VALIDATION_RATIO + config.TEST_RATIO)), random_state=config.SEED)

raw_datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df.reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df.reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df.reset_index(drop=True))
})
print(f"Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")

tokenizer = MBart50TokenizerFast.from_pretrained(config.MODEL_CHECKPOINT, src_lang=config.SOURCE_LANG_CODE, tgt_lang=config.TARGET_LANG_CODE)
model = MBartForConditionalGeneration.from_pretrained(config.MODEL_CHECKPOINT)

def preprocess_function(examples):
    model_inputs = tokenizer(examples["syl"], max_length=config.MAX_INPUT_LENGTH, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["ctg"], max_length=config.MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True, remove_columns=raw_datasets["train"].column_names)

bleu_metric = evaluate.load("sacrebleu")
meteor_metric = evaluate.load("meteor")
bertscore_metric = evaluate.load("bertscore")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    post_processed_labels = [[label.strip()] for label in decoded_labels]
    post_processed_preds = [pred.strip() for pred in decoded_preds]
    bleu = bleu_metric.compute(predictions=post_processed_preds, references=post_processed_labels)
    meteor = meteor_metric.compute(predictions=post_processed_preds, references=post_processed_labels)
    bertscore = bertscore_metric.compute(predictions=post_processed_preds, references=post_processed_labels, lang="bn")
    result = {"bleu": bleu["score"], "meteor": meteor["meteor"], "bertscore_f1": np.mean(bertscore["f1"])}
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    return {k: round(v, 4) for k, v in result.items()}

training_args = Seq2SeqTrainingArguments(
    output_dir=config.OUTPUT_DIR,
    eval_strategy=config.EVAL_STRATEGY,
    logging_strategy=config.EVAL_STRATEGY,
    save_strategy=config.EVAL_STRATEGY,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE,
    learning_rate=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY,
    num_train_epochs=config.NUM_EPOCHS,
    warmup_steps=config.WARMUP_STEPS,
    fp16=True,
    gradient_accumulation_steps=config.GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    save_total_limit=1,
    save_only_model=True,
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=tokenized_datasets["train"], eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics
)

gc.collect()
torch.cuda.empty_cache()
train_result = trainer.train()
trainer.save_model(f"{config.OUTPUT_DIR}/best_model")
tokenizer.save_pretrained(f"{config.OUTPUT_DIR}/best_model")
print(f"Best model saved to {config.OUTPUT_DIR}/best_model")

log_history = trainer.state.log_history
eval_df = pd.DataFrame([log for log in log_history if 'eval_loss' in log])
train_df_logs = pd.DataFrame([log for log in log_history if 'loss' in log and 'epoch' in log])
eval_df['epoch'] = eval_df['epoch'].round()
train_df_logs['epoch'] = train_df_logs['epoch'].round()
log_df = pd.merge(train_df_logs, eval_df, on='epoch', how='inner').drop_duplicates(subset=['epoch'], keep='last')

plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(12, 8))
plt.plot(log_df['epoch'], log_df['loss'], label='Training Loss', marker='o', linestyle='--')
plt.plot(log_df['epoch'], log_df['eval_loss'], label='Validation Loss', marker='o')
plt.title('Training & Validation Loss per Epoch (Sylhet to Chittagong)', fontsize=16, weight='bold')
plt.xlabel('Epoch', fontsize=14)
plt.ylabel('Loss', fontsize=14)
plt.legend()
plt.xticks(log_df['epoch'].unique().astype(int))
plt.grid(True)
plt.show()

fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(log_df['epoch'], log_df['eval_bleu'], label='Validation BLEU', marker='o')
ax.plot(log_df['epoch'], log_df['eval_meteor'], label='Validation METEOR', marker='s')
ax.plot(log_df['epoch'], log_df['eval_bertscore_f1'], label='Validation BERTScore-F1', marker='^')
ax.set_xlabel('Epoch', fontsize=14)
ax.set_ylabel('Score', fontsize=14)
ax.set_title('Validation Metrics per Epoch (Sylhet to Chittagong)', fontsize=16, weight='bold')
ax.legend()
ax.set_xticks(log_df['epoch'].unique().astype(int))
ax.grid(True)
plt.show()

test_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
for key, value in test_results.items():
    if key.startswith("eval_"):
        print(f"{key.replace('eval_', '').capitalize():<15}: {value:.4f}")

model_path = f"{config.OUTPUT_DIR}/best_model"
tokenizer = MBart50TokenizerFast.from_pretrained(model_path)
model = MBartForConditionalGeneration.from_pretrained(model_path).to(device)

def translate(text):
    tokenizer.src_lang = config.SOURCE_LANG_CODE
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=config.MAX_INPUT_LENGTH).to(device)
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[config.TARGET_LANG_CODE],
        max_length=config.MAX_TARGET_LENGTH, num_beams=5, early_stopping=True
    )
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

for i in range(min(10, len(test_df))):
    sample = raw_datasets['test'][i]
    source, reference = sample['syl'], sample['ctg']
    model_output = translate(source)
    print(f"Sample #{i+1}")
    print(f"  - Source (SYL):      {source}")
    print(f"  - Reference (CTG):  {reference}")
    print(f"  - Model Output (CTG): {model_output}")
    print("-" * 50)

print("\nNotebook execution complete.")